# Ressourcenverwaltung und Scheduling

Der Scheduler wählt einen geeigneten Node anhand von Ressourcen-Requests und Platzierungsregeln. Requests sind Planungsgrößen, Limits begrenzen die Nutzung. CPU-Limits können Throttling verursachen; Überschreitung des Memory-Limits kann OOMKilled auslösen.

| Mechanismus | Funktion |
|---|---|
| nodeSelector | Exakte Node-Labels |
| Node Affinity | Flexible erforderliche oder bevorzugte Node-Regeln |
| Pod Affinity / Anti-Affinity | Platzierung relativ zu anderen Pods |
| Taints / Tolerations | Nodes schließen Pods aus, sofern diese den Taint nicht tolerieren |

## Voraussetzungen

Python-Kernel, kubectl und K3s-Trainingscluster. Aus diesem Kapitelverzeichnis ausführen. Die verwendeten Demo-Ressourcen müssen frei sein. Aktiven Kontext und Namespace prüfen.

In [ ]:
%%bash
set -eo pipefail
kubectl config current-context
NS=$(kubectl config view --minify -o jsonpath='{..namespace}')
test "${NS:-default}" = default
kubectl get nodes

## 1. Nodes untersuchen

Allocatable ist die für Pods verfügbare Kapazität. Der Scheduler berücksichtigt Requests, nicht allein die aktuelle Auslastung.

In [ ]:
%%bash
set -eo pipefail
kubectl get nodes --show-labels
kubectl get nodes -o custom-columns=NAME:.metadata.name,CPU:.status.allocatable.cpu,MEMORY:.status.allocatable.memory

## 2. Scheduling-Problem erzeugen

Der Selector passt absichtlich zu keinem Node. Vorher prüfen, dass das Label nicht existiert. Erwartet: Pending und FailedScheduling.

In [ ]:
%%bash
set -eo pipefail
cat pod.yml
MATCH=$(kubectl get nodes -l training.javacream.org/scheduling=unavailable -o name)
test -z "$MATCH"
kubectl apply -f pod.yml

In [ ]:
%%bash
set -eo pipefail
kubectl get pod scheduling-demo -o wide
kubectl describe pod scheduling-demo

## 3. Problem beheben

nodeSelector eines bestehenden Pods ist nicht frei änderbar. Pod löschen und ohne den Selector neu erstellen. Keine Node-Labels oder Taints verändern.

In [ ]:
%%bash
set -eo pipefail
kubectl delete -f pod.yml
sed '/  nodeSelector:/,+1d' pod.yml | kubectl apply -f -
kubectl wait --for=condition=Ready pod/scheduling-demo --timeout=180s
kubectl get pod scheduling-demo -o wide

## 4. Affinity und Taints

requiredDuringSchedulingIgnoredDuringExecution ist eine zwingende Affinity-Regel; preferredDuringSchedulingIgnoredDuringExecution eine Präferenz. Tolerations erlauben eine Platzierung, erzwingen sie aber nicht. NoSchedule verhindert neue Platzierung, NoExecute kann bestehende Pods verdrängen. Keine Änderungen an Trainings-Nodes.

In [ ]:
%%bash
set -eo pipefail
kubectl get nodes -o custom-columns=NAME:.metadata.name,TAINTS:.spec.taints

## 5. Aufräumen

Pod entfernen.

In [ ]:
%%bash
set -eo pipefail
kubectl delete pod scheduling-demo
kubectl get pod scheduling-demo --ignore-not-found

## Fazit

Scheduling entscheidet über die Platzierung, nicht über die fachliche Hochverfügbarkeit einer Anwendung. Replikation, Ressourcen und Platzierungsregeln wirken zusammen.